# FahMai RAG Challenge — ทดสอบ LLM Local

โหลด model `typhoon2.5-qwen3-4b` จาก vLLM ที่รันบน localhost:8000

> ก่อนรัน notebook นี้ ต้องรัน vLLM ก่อน:
> ```bash
> docker compose up -d
> ```

## Step 1: ติดตั้ง Dependencies

In [3]:
# ติดตั้ง openai library สำหรับเรียก vLLM API
%pip install openai -q

/Users/sarat/Code/vllm-stack/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: เชื่อมต่อ vLLM Local

In [ ]:
from openai import OpenAI

# เชื่อมต่อ vLLM ที่รันบน localhost
# vLLM รองรับ OpenAI-compatible API จึงใช้ openai library ได้เลย
client = OpenAI(
    api_key="sk-local",                    # ค่า placeholder — vLLM ไม่ต้องการ API key จริง
    base_url="http://localhost:8000/v1",   # endpoint ของ vLLM local
)

# ชื่อโมเดลที่ใช้ใน vLLM (ชื่อเต็มจาก HuggingFace)
MODEL_NAME = "typhoon-ai/typhoon2.5-qwen3-4b"

print(f"เชื่อมต่อไปที่: http://localhost:8000/v1")
print(f"โมเดล: {MODEL_NAME}")

เชื่อมต่อไปที่: http://localhost:8000/v1
โมเดล: typhoon-ai/typhoon2.5-qwen3-4b


## Step 3: ตรวจสอบ Model ที่พร้อมใช้งาน

In [6]:
# ดูรายชื่อ model ที่ vLLM โหลดอยู่
try:
    models = client.models.list()
    print("Model ที่พร้อมใช้งาน:")
    for m in models.data:
        print(f"  - {m.id}")
except Exception as e:
    print(f"เชื่อมต่อไม่ได้: {e}")
    print("กรุณาตรวจสอบว่า vLLM รันอยู่: docker compose up -d")

Model ที่พร้อมใช้งาน:
  - typhoon-ai/typhoon2.5-qwen3-4b


## Step 4: ทดสอบ Chat Completion (Non-streaming)

In [7]:
# ทดสอบเรียก LLM แบบปกติ (รอผลครบก่อนแสดง)
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "FahMai (ฟ้าใหม่) is a fictional Thai electronics store. "
                "You are given a knowledge base of product pages, store policies, and store information (in Thai). "
                "Your task is to build a Retrieval-Augmented Generation (RAG) system that answers "
                "100 multiple-choice questions about the store. ตอบเป็นภาษาไทย"
            ),
        },
        {"role": "user", "content": "ร้านฟ้าใหม่ขายสินค้าประเภทอะไรบ้าง?"},
    ],
    temperature=0.6,
    max_completion_tokens=512,
    top_p=0.9,
)

# แสดงคำตอบ
print(response.choices[0].message.content)

ร้านฟ้าใหม่ขายสินค้าประเภทอิเล็กทรอนิกส์ต่างๆ เช่น สมาร์ทโฟน อุปกรณ์เสริมสำหรับโทรศัพท์ เช่น ชาร์จไฟ สายชาร์จ ลำโพง นาฬิกาอัจฉริยะ เครื่องใช้ไฟฟ้าในบ้าน เช่น ตู้เย็น เครื่องซักผ้า ตู้เย็นแบบประหยัดพลังงาน เครื่องปรับอากาศ เครื่องใช้ไฟฟ้าอื่นๆ รวมถึงอุปกรณ์เสริมอื่นๆ ที่เกี่ยวข้องกับชีวิตประจำวัน


## Step 5: ทดสอบ Chat Completion (Streaming)

In [8]:
# ทดสอบ streaming — แสดงผลทีละ token ขณะ LLM กำลังสร้างคำตอบ
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "FahMai (ฟ้าใหม่) is a fictional Thai electronics store. "
                "You are given a knowledge base of product pages, store policies, and store information (in Thai). "
                "Your task is to build a Retrieval-Augmented Generation (RAG) system that answers "
                "100 multiple-choice questions about the store. ตอบเป็นภาษาไทย"
            ),
        },
        {"role": "user", "content": "สวัสดี แนะนำตัวเองหน่อย"},
    ],
    temperature=0.6,
    max_completion_tokens=512,
    top_p=0.6,
    frequency_penalty=0,
    stream=True,
)

# วนรับ chunk ที่ stream มา
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()  # ขึ้นบรรทัดใหม่

สวัสดีค่ะ ผมชื่อ FahMai เป็นระบบช่วยตอบคำถามเกี่ยวกับร้านค้าอิเล็กทรอนิกส์แห่งหนึ่งในประเทศไทยค่ะ โดยมีข้อมูลเกี่ยวกับสินค้า นโยบาย และข้อมูลร้านค้ามาช่วยให้ตอบคำถามได้อย่างแม่นยำค่ะ
